Step 1 — Install packages
pip install pandas scikit-learn transformers datasets accelerate matplotlib seaborn

Step 2 — Create a new notebook

In [1]:
#Cell 1 — Imports

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

In [61]:
#Cell 2 — Check XPU

device = "xpu" if torch.xpu.is_available() else "cpu"
print("Using device:", device)
if device == "xpu":
    print("XPU device:", torch.xpu.get_device_name(0))

Using device: xpu
XPU device: Intel(R) Arc(TM) A770 Graphics


In [62]:
#Cell 3 — Load your CSV

df = pd.read_csv("../data/toxic_comments_dataset.csv")
print(df.head())
print(df.columns.tolist())
print(df.shape)
print(df["label"].unique())
print(df["label"].value_counts())

   comment_id    user_name  platform          comment_text  label language  \
0           1  user_du3mgw  Facebook     Tmi ki ba bujho 🤮  toxic  English   
1           2  user_9svp2t   YouTube       Tmi ki ba bujho  toxic  English   
2           3  user_z1h92s   YouTube  Tmi ki oa bujho... 😂  toxic  English   
3           4  user_ebzhw7   Twitter     Eta pura nonsense  toxic    Mixed   
4           5  user_g044xc    TikTok   Eta pura nonsense 😤  toxic  English   

             timestamp  length  
0  2025-09-11 06:02:18      17  
1  2025-08-03 04:12:18      15  
2  2025-08-20 22:47:18      20  
3  2025-11-01 07:34:18      17  
4  2025-08-29 02:29:18      19  
['comment_id', 'user_name', 'platform', 'comment_text', 'label', 'language', 'timestamp', 'length']
(49998, 8)
['toxic' 'severe_toxic' 'obscene' 'insult' 'threat' 'hate_speech']
label
toxic           8333
severe_toxic    8333
obscene         8333
insult          8333
threat          8333
hate_speech     8333
Name: count, dtype: int

In [63]:
#Cell 4 — Inspect columns

TEXT_COL = "comment_text"
LABEL_COL = "label"

df = df[[TEXT_COL, LABEL_COL]].copy()
df = df.dropna()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

label_list = sorted(df[LABEL_COL].unique().tolist())
print("Labels:", label_list)

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df[LABEL_COL].map(label2id)

print(df.head())
print(df["label_id"].value_counts())
print(label2id)

Labels: ['hate_speech', 'insult', 'obscene', 'severe_toxic', 'threat', 'toxic']
           comment_text  label  label_id
0     Tmi ki ba bujho 🤮  toxic         5
1       Tmi ki ba bujho  toxic         5
2  Tmi ki oa bujho... 😂  toxic         5
3     Eta pura nonsense  toxic         5
4   Eta pura nonsense 😤  toxic         5
label_id
5    8333
3    8333
2    8333
1    8333
4    8333
0    8333
Name: count, dtype: int64
{'hate_speech': 0, 'insult': 1, 'obscene': 2, 'severe_toxic': 3, 'threat': 4, 'toxic': 5}


In [64]:
print(pd.read_csv("../data/toxic_comments_dataset.csv")["label"].value_counts())

label
toxic           8333
severe_toxic    8333
obscene         8333
insult          8333
threat          8333
hate_speech     8333
Name: count, dtype: int64


In [65]:
#Cell 5 — Train-test split

train_df, test_df = train_test_split(
    df[[TEXT_COL, "label_id"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label_id"]
)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label counts:")
print(train_df["label_id"].value_counts())

Train shape: (35998, 2)
Val shape: (4000, 2)
Test shape: (10000, 2)

Train label counts:
label_id
5    6000
1    6000
0    6000
3    6000
2    5999
4    5999
Name: count, dtype: int64


In [66]:
#Cell 6 — Create Datasets

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

print(train_ds[0])

{'comment_text': 'Ki jhamela ekta!!!! 😡', 'label_id': 2}


In [67]:
#cell 7 — Load tokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded")

Tokenizer loaded


In [68]:
#cell 8 — Tokenize datasets

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COL], truncation=True)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print("Tokenization done")

Map: 100%|██████████| 10000/10000 [00:00<00:00, 48686.74 examples/s]

Tokenization done


In [69]:
#cell 9 — Set format for PyTorch

train_ds = train_ds.rename_column("label_id", "labels")
val_ds = val_ds.rename_column("label_id", "labels")
test_ds = test_ds.rename_column("label_id", "labels")

def fix_labels(example):
    example["labels"] = int(example["labels"])
    return example

train_ds = train_ds.map(fix_labels)
val_ds = val_ds.map(fix_labels)
test_ds = test_ds.map(fix_labels)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_ds[0])

Map: 100%|██████████| 10000/10000 [00:00<00:00, 33486.41 examples/s]

{'labels': tensor(2), 'input_ids': tensor([  101, 11382,  1046,  3511, 10581, 23969,  2696,   999,   999,   999,
          999,   100,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [70]:
#cell 10 — Load model

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.config.problem_type = "single_label_classification"
model = model.to(device)

print("Using device:", device)
print("Model parameter device:", next(model.parameters()).device)
print("Number of classes:", len(label_list))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9514.13it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: xpu
Model parameter device: xpu:0
Number of classes: 6


In [71]:
#cell 11 — Data collator

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Data collator ready")

Data collator ready


In [72]:
#cell 12 — Training argument

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    labels = np.array(labels).astype(int)
    preds = np.array(preds).astype(int)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [75]:
#cell 13 — Training argument

training_args = TrainingArguments(
    output_dir="./toxic_model_output",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=False,
    report_to="none",
    optim="adamw_torch",
    fp16=False,
    bf16=False,
    disable_tqdm=True
)

In [77]:
#cell 14 — Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated")

print("Trainer created")
print("Using device:", device)
print("Model parameter device:", next(model.parameters()).device)

Trainer recreated
Trainer created
Using device: xpu
Model parameter device: xpu:0


In [78]:
#cell 15 — Train

train_result = trainer.train()
print(train_result)

{'loss': '0.01891', 'grad_norm': '0.0001745', 'learning_rate': '1e-05', 'epoch': '1'}
{'eval_loss': '1.82e-06', 'eval_accuracy': '1', 'eval_f1_macro': '1', 'eval_runtime': '10.58', 'eval_samples_per_second': '378.1', 'eval_steps_per_second': '94.53', 'epoch': '1'}
{'loss': '2.273e-05', 'grad_norm': '1.779e-05', 'learning_rate': '1.111e-09', 'epoch': '2'}
{'eval_loss': '6.807e-08', 'eval_accuracy': '1', 'eval_f1_macro': '1', 'eval_runtime': '10.31', 'eval_samples_per_second': '388', 'eval_steps_per_second': '97', 'epoch': '2'}
{'train_runtime': '861.3', 'train_samples_per_second': '83.59', 'train_steps_per_second': '20.9', 'train_loss': '0.009465', 'epoch': '2'}
TrainOutput(global_step=18000, training_loss=0.009465126890275214, metrics={'train_runtime': 861.2713, 'train_samples_per_second': 83.593, 'train_steps_per_second': 20.899, 'train_loss': 0.009465126890275214, 'epoch': 2.0})


In [80]:
#cell 16 — Evaluate

test_results = trainer.predict(test_ds)

preds = np.argmax(test_results.predictions, axis=1)
labels = test_results.label_ids

preds = np.array(preds).astype(int)
labels = np.array(labels).astype(int)

print("Accuracy:", accuracy_score(labels, preds))
print("Macro F1:", f1_score(labels, preds, average="macro"))
print(classification_report(labels, preds, target_names=label_list))

Accuracy: 1.0
Macro F1: 1.0
              precision    recall  f1-score   support

 hate_speech       1.00      1.00      1.00      1666
      insult       1.00      1.00      1.00      1666
     obscene       1.00      1.00      1.00      1667
severe_toxic       1.00      1.00      1.00      1667
      threat       1.00      1.00      1.00      1667
       toxic       1.00      1.00      1.00      1667

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [81]:
#cell 17 — Save model
trainer.save_model("../models/toxic_bert_xpu")
tokenizer.save_pretrained("../models/toxic_bert_xpu")
print("Model saved successfully")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.73it/s]

Model saved successfully


In [82]:
#cell 18 — Prediction function

def predict_toxic_class(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        conf = probs[0][pred_id].item()

    return {
        "label": id2label[pred_id],
        "confidence": conf
    }

In [83]:
#cell 19 — Test prediction function

print(predict_toxic_class("You are disgusting"))
print(predict_toxic_class("I will destroy you"))
print(predict_toxic_class("Eta pura nonsense"))


{'label': 'severe_toxic', 'confidence': 0.9999998807907104}
{'label': 'severe_toxic', 'confidence': 0.9999924898147583}
{'label': 'toxic', 'confidence': 0.9999998807907104}


In [ ]:
#Cell 20 — Load saved model later
#Use this in a new notebook or later session:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = "xpu" if torch.xpu.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("./models/toxic_bert_xpu")
model = AutoModelForSequenceClassification.from_pretrained("./models/toxic_bert_xpu")
model = model.to(device)

print("Loaded model on:", device)
print(next(model.parameters()).device)

In [ ]:
#Cell 21 — Predict after loading model

def predict_toxic_class_loaded(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        conf = probs[0][pred_id].item()

    return {
        "label": model.config.id2label[pred_id],
        "confidence": conf
    }

In [ ]:
#Cell 22 — Example after reloading

print(predict_toxic_class_loaded("You are disgusting"))
print(predict_toxic_class_loaded("I will destroy you"))
print(predict_toxic_class_loaded("You are a fool."))
print(predict_toxic_class_loaded("Thank you for your help."))